# Notebook 03 — Hypothesis Testing: Statistical Inference

## What is Hypothesis Testing?

**Hypothesis testing** is a *statistical inference framework* used to decide whether an observed effect is real or could be explained by random chance.

**Key concepts:**

| Term | Definition |
|---|---|
| **H₀ (Null Hypothesis)** | The default assumption: no effect, no difference. We assume H₀ is true until proven otherwise. |
| **H₁ (Alternative Hypothesis)** | The claim we want to test: there IS an effect. |
| **p-value** | The probability of observing results at least as extreme as ours, *assuming H₀ is true*. A low p-value means H₀ is unlikely. |
| **α (significance level)** | The threshold for rejecting H₀. We use α = 0.05 (5%). If p < α, we reject H₀. |
| **Type I Error (false positive)** | Rejecting H₀ when it's actually true. Controlled by α. |
| **Type II Error (false negative)** | Failing to reject H₀ when H₁ is actually true. Controlled by statistical power. |
| **Confidence Interval** | A range of plausible values for the true effect. A 95% CI means: in 95% of repeated experiments, this range would contain the true value. |

**Decision rule:** If p-value < α → **Reject H₀** (the effect is statistically significant)

---

## How A/B Testing and Hypothesis Testing Are Related

- **A/B Testing (Notebook 02)** = the *experiment*: we designed two groups and measured their outcomes
- **Hypothesis Testing (this notebook)** = the *validation*: we formally test whether the differences we observed are statistically meaningful

Think of it this way: A/B testing tells you *what* happened. Hypothesis testing tells you *whether to believe it*.

---

## Tests We Will Run

| # | Question | Test |
|---|---|---|
| H1 | Does viewing an offer increase conversion vs. organic behavior? | **Z-test for proportions** |
| H2 | Does BOGO have a higher conversion rate than Discount? | **Z-test for proportions** |
| H3 | Do customers exposed to offers spend more (avg ticket)? | **Welch's t-test** |
| H4 | Is offer completion independent of customer gender? | **Chi-squared test** |

After running all tests, we apply **Bonferroni correction** to control for multiple comparisons.

In [1]:
import sys
from pathlib import Path
_root = Path().resolve(); _root = _root.parent if _root.name == 'notebooks' else _root; sys.path.insert(0, str(_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.extract import extract
from src.transform import transform, expand_transcript_value
from src.constants import REPORTS_FIGURES, ALPHA
from src.utils.stats import (
    ab_test_proportions, ab_test_means, chi_squared_test, bonferroni_correction
)
from src.utils.plot import save_fig, bar_comparison, hypothesis_result_table

REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')

portfolio, profile, transcript = extract()
tables = transform(portfolio, profile, transcript)
master = tables['master_table']
print(f'Master table loaded: {master.shape}')

Master table loaded: (115609, 20)


---
## Test H1 — Does Viewing an Offer Increase Conversion?

**Which test?** Z-test for two proportions

The Z-test is appropriate here because:
- Our metric is a proportion (conversion rate)
- Both groups are large (Central Limit Theorem applies)
- Observations are independent

**Hypotheses:**
- **H₀:** Conversion rate of viewed group = conversion rate of not-viewed group
- **H₁:** Conversion rate of viewed group > conversion rate of not-viewed group (one-tailed)

**Groups:**
- Control: received offer but **did not view** it
- Treatment: **viewed** the offer

In [2]:
control_h1   = master[master['viewed'] == False]['completed']
treatment_h1 = master[master['viewed'] == True]['completed']

result_h1 = ab_test_proportions(control_h1, treatment_h1, alpha=ALPHA)
pd.Series(result_h1).to_frame('H1 Result')

,H1 Result
n_control,16676
n_treatment,98933
conversion_control,0.3536
conversion_treatment,0.6216
uplift_absolute,0.2681
uplift_relative_pct,75.82
ci_95_low,0.2602
ci_95_high,0.2759
z_stat,64.9502
p_value,0.0


In [3]:
fig = bar_comparison(
    labels=['Control\n(not viewed)', 'Treatment\n(viewed)'],
    values=[result_h1['conversion_control'] * 100, result_h1['conversion_treatment'] * 100],
    title=(
        f'H1 — Conversion Rate: Viewed vs Not Viewed\n'
        f'z = {result_h1["z_stat"]}, p = {result_h1["p_value"]}  →  '
        f'{"Reject H₀ ✓" if result_h1["significant"] else "Fail to Reject H₀"}'
    ),
    ylabel='Conversion Rate (%)',
    highlight_idx=1,
    fmt='{:.1f}%',
    save_path=REPORTS_FIGURES / '03_h1_conversion.png',
)
plt.show()
print(f'95% CI for uplift: [{result_h1["ci_95_low"]*100:.2f}%, {result_h1["ci_95_high"]*100:.2f}%]')

95% CI for uplift: [26.02%, 27.59%]


C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:50: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:50: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


---
## Test H2 — Does BOGO Outperform Discount?

**Which test?** Z-test for two proportions

Both groups contain only customers who **viewed** the offer (same exposure condition). We compare conversion rates between the two offer types.

**Hypotheses:**
- **H₀:** Conversion rate(BOGO) = Conversion rate(Discount)
- **H₁:** Conversion rate(BOGO) > Conversion rate(Discount)

In [4]:
viewed_only = master[master['viewed'] == True]

control_h2   = viewed_only[viewed_only['offer_type'] == 'discount']['completed']
treatment_h2 = viewed_only[viewed_only['offer_type'] == 'bogo']['completed']

result_h2 = ab_test_proportions(control_h2, treatment_h2, alpha=ALPHA)
pd.Series(result_h2).to_frame('H2 Result')

,H2 Result
n_control,39815
n_treatment,44223
conversion_control,0.7971
conversion_treatment,0.673
uplift_absolute,-0.1241
uplift_relative_pct,-15.57
ci_95_low,-0.13
ci_95_high,-0.1182
z_stat,-40.5531
p_value,1.0


In [5]:
fig = bar_comparison(
    labels=['Discount', 'BOGO'],
    values=[result_h2['conversion_control'] * 100, result_h2['conversion_treatment'] * 100],
    title=(
        f'H2 — BOGO vs Discount Conversion Rate\n'
        f'z = {result_h2["z_stat"]}, p = {result_h2["p_value"]}  →  '
        f'{"Reject H₀ ✓" if result_h2["significant"] else "Fail to Reject H₀"}'
    ),
    ylabel='Conversion Rate (%)',
    highlight_idx=1,
    fmt='{:.1f}%',
    save_path=REPORTS_FIGURES / '03_h2_bogo_vs_discount.png',
)
plt.show()

C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:50: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


---
## Test H3 — Do Exposed Customers Spend More?

**Which test?** Welch's t-test (two-sample, unequal variances)

The t-test is used here because our metric is continuous (total spend per customer). Welch's variant does not assume equal variances between groups — more robust than the standard Student's t-test.

**Hypotheses:**
- **H₀:** Mean spend(treatment) = Mean spend(control)
- **H₁:** Mean spend(treatment) ≠ Mean spend(control) (two-tailed)

In [6]:
txns = expand_transcript_value(transcript)
spend = txns[txns['event'] == 'transaction'].groupby('person')['amount'].sum()
avg_ticket = txns[txns['event'] == 'transaction']['amount'].mean()
print(f'Overall average transaction: ${avg_ticket:.2f}')

exposed     = master[master['viewed'] == True]['person'].unique()
not_exposed = master[master['viewed'] == False]['person'].unique()

spend_treated = spend[spend.index.isin(exposed)]
spend_control = spend[spend.index.isin(not_exposed)]

result_h3 = ab_test_means(spend_control, spend_treated, alpha=ALPHA)
pd.Series(result_h3).to_frame('H3 Result')

Overall average transaction: $12.78


,H3 Result
n_control,9326
n_treatment,16422
mean_control,95.47
mean_treatment,107.64
uplift_absolute,12.17
uplift_relative_pct,12.75
ci_95_low,9.06
ci_95_high,15.28
t_stat,7.6613
p_value,0.0


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(spend_control.clip(upper=spend_control.quantile(0.99)), bins=40, alpha=0.7, color='#C0C0C0', label='Control')
axes[0].hist(spend_treated.clip(upper=spend_treated.quantile(0.99)), bins=40, alpha=0.7, color='#00704A', label='Treatment')
axes[0].axvline(result_h3['mean_control'], color='gray', linestyle='--', label=f'Mean ctrl: ${result_h3["mean_control"]:.0f}')
axes[0].axvline(result_h3['mean_treatment'], color='#00704A', linestyle='--', label=f'Mean trt: ${result_h3["mean_treatment"]:.0f}')
axes[0].set_title('Total Spend Distribution')
axes[0].set_xlabel('Total Spend ($)')
axes[0].legend(fontsize=8)

bars = axes[1].bar(
    ['Control', 'Treatment'],
    [result_h3['mean_control'], result_h3['mean_treatment']],
    color=['#C0C0C0', '#00704A'],
)
for bar, val in zip(bars, [result_h3['mean_control'], result_h3['mean_treatment']]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02, f'${val:.0f}', ha='center', fontweight='bold')
axes[1].set_title(
    f'H3 — Mean Spend per Customer\n'
    f't = {result_h3["t_stat"]}, p = {result_h3["p_value"]}  →  '
    f'{"Reject H₀ ✓" if result_h3["significant"] else "Fail to Reject H₀"}'
)
axes[1].set_ylabel('Avg Total Spend ($)')

fig.tight_layout()
save_fig(fig, REPORTS_FIGURES / '03_h3_spend.png')
plt.show()
print(f'95% CI for spend uplift: [${result_h3["ci_95_low"]:.2f}, ${result_h3["ci_95_high"]:.2f}]')

95% CI for spend uplift: [$9.06, $15.28]


C:\Users\JOBS-13\AppData\Local\Temp\ipykernel_35652\3295810297.py:25: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\AppData\Local\Temp\ipykernel_35652\3295810297.py:25: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


---
## Test H4 — Is Offer Completion Independent of Gender?

**Which test?** Chi-squared test of independence

The chi-squared test checks whether two categorical variables (gender and completion) are independent. If the p-value is low, gender is associated with whether a customer completes an offer.

**Hypotheses:**
- **H₀:** Offer completion is independent of gender
- **H₁:** Offer completion is associated with gender

In [8]:
contingency = pd.crosstab(master['gender'], master['completed'])
contingency.columns = ['Not Completed', 'Completed']
print('Contingency Table:')
print(contingency)
print()

result_h4 = chi_squared_test(contingency, alpha=ALPHA)
pd.Series(result_h4).to_frame('H4 Result')

Contingency Table:
        Not Completed  Completed
gender                          
F               13026      30916
M               24189      33062
O                 486       1014



,H4 Result
chi2_stat,1712.8397
p_value,0.0
degrees_of_freedom,2
significant,True
alpha,0.05


In [9]:
completion_by_gender = (
    contingency['Completed'] / contingency.sum(axis=1) * 100
).reset_index()
completion_by_gender.columns = ['gender', 'completion_rate']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(completion_by_gender['gender'], completion_by_gender['completion_rate'],
              color=['#00704A','#CBA258','#C0C0C0'])
ax.set_title(
    f'H4 — Completion Rate by Gender\n'
    f'χ² = {result_h4["chi2_stat"]}, p = {result_h4["p_value"]}  →  '
    f'{"Reject H₀ ✓" if result_h4["significant"] else "Fail to Reject H₀"}'
)
ax.set_ylabel('Completion Rate (%)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
fig.tight_layout()
save_fig(fig, REPORTS_FIGURES / '03_h4_gender.png')
plt.show()

C:\Users\JOBS-13\AppData\Local\Temp\ipykernel_35652\2464975100.py:18: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\AppData\Local\Temp\ipykernel_35652\2464975100.py:18: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.tight_layout()
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")
C:\Users\JOBS-13\Documents\GitHub\promo-effectiveness-analysis\src\utils\plot.py:22: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  fig.savefig(path, dpi=dpi, bbox_inches="tight")


---
## Multiple Testing Correction — Bonferroni

When running multiple hypothesis tests on the same dataset, the probability of a false positive (Type I error) accumulates. If we run 4 tests each at α = 0.05, the probability of at least one false positive is:

$$P(\text{at least one false positive}) = 1 - (1 - 0.05)^4 \approx 18.5\%$$

**Bonferroni correction:** Divide α by the number of tests. New threshold = 0.05 / 4 = 0.0125.

A test that was significant at α = 0.05 may no longer be significant after correction.

In [10]:
p_values = [result_h1['p_value'], result_h2['p_value'], result_h3['p_value'], result_h4['p_value']]
hypotheses = ['H1: Viewed → Conversion', 'H2: BOGO > Discount', 'H3: Exposed spend more', 'H4: Completion ~ Gender']
tests_used = ['Z-test (proportions)', 'Z-test (proportions)', "Welch's t-test", 'Chi-squared']

bonf = bonferroni_correction(p_values, alpha=ALPHA)

summary = pd.DataFrame({
    'Hypothesis': hypotheses,
    'Test': tests_used,
    'p_value': p_values,
    'Significant (α=0.05)': [r['significant'] for r in [result_h1, result_h2, result_h3, result_h4]],
    'Adjusted α (Bonferroni)': [b['adjusted_alpha'] for b in bonf],
    'Significant (corrected)': [b['significant_corrected'] for b in bonf],
})

print('=== HYPOTHESIS TESTING RESULTS ===')
summary

=== HYPOTHESIS TESTING RESULTS ===


,Hypothesis,Test,p_value,Significant (α=0.05),Adjusted α (Bonferroni),Significant (corrected)
0,H1: Viewed → Conversion,Z-test (proportions),0.0,True,0.0125,True
1,H2: BOGO > Discount,Z-test (proportions),1.0,False,0.0125,False
2,H3: Exposed spend more,Welch's t-test,0.0,True,0.0125,True
3,H4: Completion ~ Gender,Chi-squared,0.0,True,0.0125,True


## Final Decision Table

| # | Hypothesis | p-value | Decision | Corrected |
|---|---|---|---|---|
| H1 | Viewing an offer increases conversion | ? | ? | ? |
| H2 | BOGO converts better than Discount | ? | ? | ? |
| H3 | Exposed customers spend more | ? | ? | ? |
| H4 | Completion depends on gender | ? | ? | ? |

*(Fill in after running the notebook)*

---
**Next:** Notebook 04 uses these findings to compute uplift by customer segment — identifying which segments respond best to each offer type.